# QuestionGen UI Launcher

Primary staff-facing Colab launcher. This notebook mounts Drive, loads secrets, installs the selected pushed branch from a notebook-side allowlist, and launches the Gradio app immediately.

Choose between CSV upload and a Drive CSV path inside the Gradio UI itself. `REPO_BRANCH` must come from `REPO_BRANCH_OPTIONS`, which currently allows only `main` and defaults to `main`. Unpushed local-only workspace changes cannot be pulled into Colab automatically.


## 1. Mount Drive And Define Standard Paths


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

# Edit DATA_DIR only if your runtime folder lives somewhere else in Drive.
DATA_DIR = Path("/content/drive/MyDrive/Work/Ebenezer Related/QuestionGenData")
SECRETS_DIR = DATA_DIR / "secrets"
INPUT_DIR = DATA_DIR / "input"
OUTPUT_DIR = DATA_DIR / "output"
LOGS_DIR = DATA_DIR / "logs"

for folder in [SECRETS_DIR, INPUT_DIR, OUTPUT_DIR, LOGS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR)
print("SECRETS_DIR:", SECRETS_DIR)
print("INPUT_DIR:", INPUT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


## 2. Minimal Settings


In [ ]:
API_KEY_FILE = SECRETS_DIR / "api_key.txt"
DEFAULT_DRIVE_CSV = INPUT_DIR / "questions.csv"
GRADIO_OUTPUT_DIR = OUTPUT_DIR / "gradio"

print("API_KEY_FILE:", API_KEY_FILE)
print("Default Drive CSV shown in UI:", DEFAULT_DRIVE_CSV)
print("Default Gradio output dir shown in UI:", GRADIO_OUTPUT_DIR)


## 3. Load Secrets From Drive


In [ ]:
import os
from pathlib import Path

def load_api_keys(filepath: str | Path) -> None:
    filepath = Path(filepath)

    if not filepath.exists():
        raise FileNotFoundError(
            f"API key file not found: {filepath}\n"
            "Create it with lines like OPENAI_API_KEY=sk-..."
        )

    with filepath.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()

            if not line or line.startswith("#") or "=" not in line:
                continue

            key, value = line.split("=", 1)
            os.environ[key.strip()] = value.strip()

load_api_keys(API_KEY_FILE)

MODEL_NAME = os.getenv("QUESTIONGEN_MODEL", "gpt-5-mini")
TEMPERATURE = float(os.getenv("QUESTIONGEN_TEMPERATURE", "0"))

print("Loaded env vars:")
print("- OPENAI_API_KEY:", "set" if os.getenv("OPENAI_API_KEY") else "missing")
print("- QUESTIONGEN_MODEL:", MODEL_NAME)
print("- QUESTIONGEN_TEMPERATURE:", TEMPERATURE)


## 4. Advanced Settings

`REPO_BRANCH` must be one of the pushed branches listed in `REPO_BRANCH_OPTIONS`. The active launcher currently defaults to `main` and allows only `main` unless you intentionally broaden the allowlist. Colab can pull pushed branches, but it cannot see unpushed local-only edits from this workspace.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/AwkAwkAardvark/QuestionGen.git"
REPO_BRANCH_OPTIONS = [
    "main",
]
REPO_BRANCH = REPO_BRANCH_OPTIONS[0]
REPO_DIR = Path("/content/QuestionGen")

if REPO_BRANCH not in REPO_BRANCH_OPTIONS:
    raise ValueError(
        f"REPO_BRANCH must be one of {REPO_BRANCH_OPTIONS}. "
        "Update REPO_BRANCH_OPTIONS only when you intentionally want Colab to target another pushed branch."
    )

print("REPO_URL:", REPO_URL)
print("REPO_BRANCH_OPTIONS:", REPO_BRANCH_OPTIONS)
print("REPO_BRANCH:", REPO_BRANCH)
print("REPO_DIR:", REPO_DIR)


## 5. Clone, Install, And Launch Gradio

This notebook does not run `run_batch_files(...)` directly before launch. The Gradio app is the active UI surface, and it handles upload-vs-Drive-path input selection inside the app itself.


In [ ]:
import importlib
import importlib.util
import os
import shutil
import subprocess
import sys

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
    check=True,
)
get_ipython().run_line_magic("pip", f"install -e '{REPO_DIR}[ui]'")

SRC_DIR = REPO_DIR / "src"
importlib.invalidate_caches()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

if importlib.util.find_spec("questiongen") is None:
    raise ModuleNotFoundError(
        "questiongen is still not importable after installation. "
        "Check the install cell output before continuing."
    )

os.environ["QUESTIONGEN_DATA_DIR"] = str(DATA_DIR)
os.environ["QUESTIONGEN_API_KEY_PATH"] = str(API_KEY_FILE)
os.environ["QUESTIONGEN_DRIVE_INPUT_CSV"] = str(DEFAULT_DRIVE_CSV)
os.environ["QUESTIONGEN_OUTPUT_DIR"] = str(GRADIO_OUTPUT_DIR)

print("Cloned and installed branch:", REPO_BRANCH)
print("Repo dir:", REPO_DIR)
print("Source path added:", SRC_DIR)
print("Python executable:", sys.executable)

from questiongen.ui.gradio_app import create_app

app = create_app()
app.launch(debug=True)
